In [16]:
import pandas as pd
import hashlib
import json
from pathlib import Path
from datetime import datetime

In [15]:
import shutil
from pathlib import Path
from datetime import datetime

# Paths
HISTORY_PATH = Path("selection_history.json")
TRAIN_DIR    = Path("training_sets")
OUTPUT_DIR   = Path(".")

# Backup folder with timestamp
BACKUP_DIR = Path("backup_before_reset") / datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# Move history file if present
if HISTORY_PATH.exists():
    shutil.move(str(HISTORY_PATH), BACKUP_DIR / HISTORY_PATH.name)

# Move previous training sets
if TRAIN_DIR.exists():
    shutil.move(str(TRAIN_DIR), BACKUP_DIR / TRAIN_DIR.name)

# Move any unique_sample_*.csv files
moved_any = False
for p in OUTPUT_DIR.glob("unique_sample_*.csv"):
    shutil.move(str(p), BACKUP_DIR / p.name)
    moved_any = True

print("✅ Reset complete. Archived prior state to:", BACKUP_DIR.resolve())


✅ Reset complete. Archived prior state to: /home/ubuntu/tw_rp_main/jupyter-notebooks/backup_before_reset/20251002_042945


In [17]:
def row_hash(row: pd.Series) -> str:
    """Generate a deterministic hash of a row if no explicit ID column exists."""
    obj = row.to_dict()
    normalized = {str(k): ("" if pd.isna(v) else str(v)) for k, v in obj.items()}
    payload = json.dumps(normalized, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def load_df(path: str, dataset_name: str, id_column: str = None) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df["__dataset"] = dataset_name
    df["__source_file"] = Path(path).name

    if id_column and id_column in df.columns:
        df["unique_key"] = df[id_column].astype(str)
    else:
        df["unique_key"] = df.apply(row_hash, axis=1)

    return df


In [18]:
# Update these paths to your local copies
ABORTION_PATH = "/home/ubuntu/tw_rp_main/datasets/abortion_data-updated - new_abortion_related_subreddits_text_posts .csv"
MISCARRIAGE_PATH = "/home/ubuntu/tw_rp_main/datasets/miscarriage-data-updated - miscarriage_related_posts.csv"
HARASSMENT_PATH = "/home/ubuntu/tw_rp_main/datasets/sexual-harrassment-data-updated - RelevantByTitle.csv"

# History file (persists across runs)
HISTORY_PATH = Path("selection_history.json")

# Desired counts per dataset
counts = {
    "abortion": 167,
    "miscarriage": 167,
    "harassment": 166
}

# Optional: if your CSVs have a post_id or id column
ID_COLUMN = None   # e.g. "post_id"


In [19]:
# Load CSVs
abortion_df = load_df(ABORTION_PATH, "abortion", ID_COLUMN)
miscarriage_df = load_df(MISCARRIAGE_PATH, "miscarriage", ID_COLUMN)
harassment_df = load_df(HARASSMENT_PATH, "harassment", ID_COLUMN)

# Load or initialize selection history
if HISTORY_PATH.exists():
    with open(HISTORY_PATH, "r", encoding="utf-8") as f:
        history = json.load(f)
else:
    history = {"used_keys": [], "runs": []}

used_keys = set(history.get("used_keys", []))

# Exclude previously used posts
def exclude_used(df):
    return df[~df["unique_key"].isin(used_keys)].copy()

ab_pool = exclude_used(abortion_df)
mi_pool = exclude_used(miscarriage_df)
sh_pool = exclude_used(harassment_df)

In [20]:
shortages = []
if len(ab_pool) < counts["abortion"]:
    shortages.append(f"abortion (need {counts['abortion']}, have {len(ab_pool)})")
if len(mi_pool) < counts["miscarriage"]:
    shortages.append(f"miscarriage (need {counts['miscarriage']}, have {len(mi_pool)})")
if len(sh_pool) < counts["harassment"]:
    shortages.append(f"harassment (need {counts['harassment']}, have {len(sh_pool)})")

if shortages:
    raise RuntimeError("Not enough fresh rows: " + "; ".join(shortages))

sample_ab = ab_pool.sample(n=counts["abortion"], replace=False, random_state=None)
sample_mi = mi_pool.sample(n=counts["miscarriage"], replace=False, random_state=None)
sample_sh = sh_pool.sample(n=counts["harassment"], replace=False, random_state=None)

sample_all = pd.concat([sample_ab, sample_mi, sample_sh], ignore_index=True)
sample_all = sample_all.sample(frac=1.0).reset_index(drop=True)  # shuffle


In [21]:
# Save timestamped CSV
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = Path(f"unique_sample_{ts}.csv")
sample_all.to_csv(out_path, index=False)

# Update history
new_keys = sample_all["unique_key"].tolist()
history["used_keys"].extend(new_keys)
history["runs"].append({
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "output_file": str(out_path),
    "counts": counts,
    "selected": len(new_keys)
})

with open(HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(sample_all)} posts to {out_path}")
print(f"Remaining after this run:")
print("  abortion:", len(ab_pool) - counts["abortion"])
print("  miscarriage:", len(mi_pool) - counts["miscarriage"])
print("  harassment:", len(sh_pool) - counts["harassment"])


✅ Saved 500 posts to unique_sample_20251002_043003.csv
Remaining after this run:
  abortion: 4244
  miscarriage: 819
  harassment: 4994


In [22]:
# Make a clean training label column (good for ML pipelines)
sample_all = sample_all.copy()
sample_all["label"] = sample_all["__dataset"]  # keep your original columns intact

# Create a training_sets folder
TRAIN_DIR = Path("training_sets")
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

# Save a per-run training file (500 rows)
train_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
train_csv = TRAIN_DIR / f"train_{train_ts}.csv"
sample_all.to_csv(train_csv, index=False)

# Also keep a stable "latest" pointer you can reference in code
latest_csv = TRAIN_DIR / "train_latest.csv"
sample_all.to_csv(latest_csv, index=False)

print(f"✅ Saved training set (500 rows): {train_csv}")
print(f"🔁 Also updated: {latest_csv}")

# (Optional) Save per-class training files for class-specific experiments
PER_CLASS_DIR = TRAIN_DIR / f"per_class_{train_ts}"
PER_CLASS_DIR.mkdir(parents=True, exist_ok=True)

for cls in sample_all["label"].unique():
    out_cls = PER_CLASS_DIR / f"{cls}_train_{train_ts}.csv"
    sample_all[sample_all["label"] == cls].to_csv(out_cls, index=False)
    print(f"• Saved {cls} subset to: {out_cls}")

# (Optional) Keep a cumulative union of everything ever sampled (good for audit/repro)
CUMULATIVE_CSV = TRAIN_DIR / "all_selected_so_far.csv"
if CUMULATIVE_CSV.exists():
    prev = pd.read_csv(CUMULATIVE_CSV, low_memory=False)
    # Use unique_key to de-dup
    combined = pd.concat([prev, sample_all], ignore_index=True)
    combined = combined.drop_duplicates(subset=["unique_key"])
else:
    combined = sample_all

combined.to_csv(CUMULATIVE_CSV, index=False)
print(f"📚 Cumulative selected-so-far updated: {CUMULATIVE_CSV}")


✅ Saved training set (500 rows): training_sets/train_20251002_043005.csv
🔁 Also updated: training_sets/train_latest.csv
• Saved miscarriage subset to: training_sets/per_class_20251002_043005/miscarriage_train_20251002_043005.csv
• Saved abortion subset to: training_sets/per_class_20251002_043005/abortion_train_20251002_043005.csv
• Saved harassment subset to: training_sets/per_class_20251002_043005/harassment_train_20251002_043005.csv
📚 Cumulative selected-so-far updated: training_sets/all_selected_so_far.csv


In [23]:
sample_all.head(10)

,id,subreddit,title,selftext,created_utc,url,Tags,__dataset,__source_file,unique_key,label
0,on02g1,Pregnant,Gender disappointment somewhat. FTM with a BOY...,"I know this is a thing, and quite common, but ...",2021-07-18 21:41:32,https://www.reddit.com/r/pregnant/comments/on0...,NaN,miscarriage,miscarriage-data-updated - miscarriage_related...,a7e3e79c6f5b569baba86044ca464306989604171d3def...,miscarriage
1,18p5rk7,BabyBumps,Anxiety- miscarriage concerns,I run anxious but I’ve been 400% more anxious ...,2023-12-23 13:32:35,https://www.reddit.com/r/BabyBumps/comments/18...,NaN,miscarriage,miscarriage-data-updated - miscarriage_related...,932e260efe87a6d3ddddaaf44defe2ef6e987dea5d3a83...,miscarriage
2,wgoybd,abortion,"I regret my abortion, but I remain pro-choice.",Maybe it was because my decision was so rushed...,2022-08-05 7:25:39,https://www.reddit.com/r/abortion/comments/wgo...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,6bac8a6d3ad3fabbde5107e91d79f5f5ac000eb86fa9fe...,abortion
3,1l494t2,depression,i hate being trans,it's so hard because you know that no one will...,2025-06-05 20:05:23,https://www.reddit.com/r/depression/comments/1...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,167f89b1806a5a3e24062ced1d31bd7293b5484cea65bd...,abortion
4,oj3t14,assault,looking for help years after abuse,lately i’ve been feeling trapped by the abuse ...,2021-07-13 0:01:48,https://www.reddit.com/r/sexualassault/comment...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,4fe84fcae64aa1a3d67892abed43dedaffcae3d82ff418...,harassment
5,1kze5ws,Miscarriage,miscarriage 5 weeks,Wondering if anyone has ever experienced somet...,2025-05-30 20:11:35,https://www.reddit.com/r/Miscarriage/comments/...,NaN,miscarriage,miscarriage-data-updated - miscarriage_related...,250036dc985f4f57f6a6609793aaac7811f80cbd3360df...,miscarriage
6,1kg6kg5,abortion,My baby daddy threatens me he will unalive him...,I was not planning on letting him know but my ...,2025-05-06 14:57:10,https://www.reddit.com/r/abortion/comments/1kg...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,81124744c03eaf80278071d8890d52d6532fc3a88fff14...,abortion
7,czt1x8,abortion,Abortion Doula Available Edmonton Area,I am an abortion doula. I support patients as ...,2019-09-05 0:02:20,https://www.reddit.com/r/abortion/comments/czt...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,60f4f8dd8286b58a7f3d74433bb12ec727633bb2eeb753...,abortion
8,1lhvhi8,Miscarriage,Trying again after miscarriage,Hello. Hoping people can give me their opinion...,2025-06-22 18:51:48,https://www.reddit.com/r/Miscarriage/comments/...,NaN,miscarriage,miscarriage-data-updated - miscarriage_related...,5ac6024d272f1bab68a539a188e4d1f2fa506c9d17ad75...,miscarriage
9,1lf1jo0,abortion,Vomiting after swallowing the misoprostol tablets,After 36 hours of taking the first pill (mifep...,2025-06-19 4:12:49,https://www.reddit.com/r/abortion/comments/1lf...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,a07f8d96b0bc20def961701d8ec4cbd0924184b6d0a6b4...,abortion
